In [ ]:
import numpy as np
import pandas as pd
tweet= pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test=pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
df=pd.concat([tweet,test])

文本清理函数：移除URL HTML标签 表情符号 标点符号

In [ ]:
import re
import string

def clean_text(text):
    if not isinstance(text, str):
        return text
    
    # 1. 移除URL
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    text = url_pattern.sub(r'', text)
    
    # 2. 移除HTML标签
    html_pattern = re.compile(r'<.*?>')
    text = html_pattern.sub(r'', text)
    
    # 3. 移除表情符号
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # 4. 移除标点符号
    table = str.maketrans('', '', string.punctuation)
    text = text.translate(table)
    
    return text

df['text'] = df['text'].apply(lambda x: clean_text(x))

对每条tweet进行分词处理word_tokenize(tweet)；

将所有单词转换为小写word.lower()；

将处理后的单词列表添加到语料库corpus中

In [ ]:
from tqdm import tqdm
from nltk.tokenize import word_tokenize
def create_corpus_new(df):
    corpus=[]
    for tweet in tqdm(df['text']):
        words=[word.lower() for word in word_tokenize(tweet)]
        corpus.append(words)
    return corpus
corpus=create_corpus_new(df)

根据GloVe 构建embedding_dict字典；词->向量

文件中第一个元素为词，后面的元素为对应向量

In [ ]:
embedding_dict={}
with open('/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.100d.txt','r') as f:
    for line in f:
        values=line.split()
        word = values[0]
        vectors=np.asarray(values[1:],'float32')
        embedding_dict[word]=vectors
f.close()

Tokenizer()根据语料库corpus构建词汇表；

将文本转换为数值序列 并填充为相同的长度tweet_pad

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 50
tokenizer_obj = Tokenizer()
tokenizer_obj.fit_on_texts(corpus)
sequences = tokenizer_obj.texts_to_sequences(corpus)

tweet_pad = pad_sequences(sequences, maxlen=MAX_LEN, truncating='post', padding='post')
tweet_pad[0][0:] # 第一句话示例

tweet中词汇表每个词 构建embedding_matrix

In [ ]:
word_index=tokenizer_obj.word_index # 词汇表词个数
num_words=len(word_index)+1
embedding_matrix=np.zeros((num_words,100)) # 每个词 长度为100的向量

for word,i in tqdm(word_index.items()):
    if i < num_words:
        emb_vec=embedding_dict.get(word)
        if emb_vec is not None: # 在GloVe中有对应向量 就填入
            embedding_matrix[i]=emb_vec

双向LSTM文本分类模型：Embedding + Dropout + Bidirectional LSTM + Dense Layer + Output Layer

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional, Dropout
from tensorflow.keras.initializers import Constant
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

model = Sequential()

# 嵌入层 embedding_matrix
embedding = Embedding(
    num_words, 
    100,
    embeddings_initializer=Constant(embedding_matrix),
    trainable=False  # 保持预训练词向量不变
)
model.add(embedding)

# 空间dropout
model.add(SpatialDropout1D(0.3))

# 双向LSTM
model.add(Bidirectional(LSTM(
    64,
    dropout=0.3, 
    recurrent_dropout=0.3,
    return_sequences=False  # 只返回最后输出
)))

# 添加全连接层
model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.001)))
model.add(Dropout(0.5))

# 输出层
model.add(Dense(1, activation='sigmoid'))

# 优化器
optimizer = Adam(
    learning_rate=1e-4,  # 降低学习率
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7
)

# 编译模型
model.compile(
    loss='binary_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy', 'precision', 'recall']
)

model.build(input_shape=(None, MAX_LEN))
model.summary()

划分训练数据+训练

In [ ]:
from sklearn.model_selection import train_test_split
train=tweet_pad[:tweet.shape[0]]
test=tweet_pad[tweet.shape[0]:]
X_train,X_test,y_train,y_test=train_test_split(train,tweet['target'].values,test_size=0.2)
print('Shape of train',X_train.shape)
print("Shape of Validation ",X_test.shape)
history=model.fit(X_train,y_train,batch_size=4,epochs=15,validation_data=(X_test,y_test),verbose=2)

预测+提交

In [ ]:
pred_GloVe = model.predict(test)
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sample_submission["target"] = pred_GloVe.round().astype('int')  # 使用LSTM预测结果
sample_submission.to_csv("submission.csv", index=False)